In [13]:
# Multiple ways to get and work with home directory
import os
from pathlib import Path
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity
from torch.nn.functional import cosine_similarity as torch_cosine_similarity




# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
#print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Similar Papers"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi
Current working directory: /Users/naveendeevi/Similar Papers


In [14]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/naveendeevi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/naveendeevi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [59]:



def simple_similarity(sent1: str, sent2: str) -> float:
    """
    Computes simple similarity between two sentences after:
    - Removing OpenAlex-style IDs (W##########)
    - Removing stop words
    - Applying stemming
    - Using Jaccard similarity on resulting tokens

    Returns:
        float: similarity score between 0 and 1
    """
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()

    def clean(sentence):
        # Remove OpenAlex-style IDs
        sentence = re.sub(r'\bW\d{10}\b', '', sentence)

        # Lowercase and remove punctuation
        sentence = re.sub(r'[^\w\s]', '', sentence.lower())

        # Tokenize, remove stopwords, stem
        tokens = word_tokenize(sentence)
        filtered = [stemmer.stem(w) for w in tokens if w not in stop_words]
        return set(filtered)

    tokens1 = clean(sent1)
    tokens2 = clean(sent2)

    if not tokens1 or not tokens2:
        return 0.0

    # Jaccard similarity: intersection / union
    intersection = tokens1 & tokens2
    union = tokens1 | tokens2
    return len(intersection) / len(union)

# Word cloud & Sentence similarity
def word_cloud_jaccard_similarity(word_list, sentence):
    """
    Compute Jaccard similarity between a list of words and a sentence.

    Args:
        word_list (list): A list of words (e.g., MDD_Word_Cloud)
        sentence (str): A sentence to compare with the word list

    Returns:
        float: Jaccard similarity score (0.0 to 1.0)
    """
    if not isinstance(sentence, str) or not sentence.strip():
        return 0.0

    # Convert both to sets
    set1 = set(w.lower() for w in word_list)
    set2 = set(sentence.lower().split())

    # Compute Jaccard index
    intersection = set1.intersection(set2)
    union = set1.union(set2)

    if not union:
        return 0.0

    return len(intersection) / len(union)




def compute_tfidf_scores_apply(df, ref_col, cmp_col, score_col='TFIDF_Score'):
    """
    Compute TF-IDF similarity scores group-wise and map back using .apply().
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        ref_col (str): Column name for reference sentence.
        cmp_col (str): Column name for comparison sentence.
        score_col (str): Name of the new score column.
    
    Returns:
        pd.DataFrame: Same DataFrame with a new column `score_col` for TF-IDF similarity.
    """
    stemmer = PorterStemmer()

    def clean(text):
        text = str(text)
        text = re.sub(r'[^\w\s]', '', text.lower())
        return ' '.join([stemmer.stem(word) for word in text.split() if word not in ENGLISH_STOP_WORDS])

    # Copy & clean
    df = df.copy()
    df['clean_ref'] = df[ref_col].fillna('').apply(clean)
    df['clean_cmp'] = df[cmp_col].fillna('').apply(clean)

    # Identify reference groups (same ref sentence = same group)
    df['ref_group'] = (df[ref_col] != df[ref_col].shift()).cumsum()

    # Dict to store all similarity scores
    similarity_lookup = {}

    for group_id, group_df in df.groupby('ref_group'):
        ref_text = group_df['clean_ref'].iloc[0]
        cmp_texts = group_df['clean_cmp'].tolist()
        corpus = [ref_text] + cmp_texts

        tfidf = TfidfVectorizer()
        mat = tfidf.fit_transform(corpus)

        sim_scores = sklearn_cosine_similarity(mat[0:1], mat[1:])[0]

        for idx, original_index in enumerate(group_df.index):
            similarity_lookup[original_index] = sim_scores[idx]

    # Apply using lookup
    df[score_col] = df.index.to_series().map(similarity_lookup)

    # Drop helper columns
    return df.drop(columns=['clean_ref', 'clean_cmp', 'ref_group'])


# Load SBERT model (compact and performant)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Row-wise similarity function
def sbert_row_similarity(row, ref_col, cmp_col):
    if pd.isna(row[ref_col]) or pd.isna(row[cmp_col]):
        return np.nan
    emb1 = model.encode(row[ref_col], convert_to_tensor=True)
    emb2 = model.encode(row[cmp_col], convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

# Corpus-wide SBERT Similarity (One-to-Many)
def sbert_group_similarity(df, ref_col, cmp_col, score_col='SBERT_group_similarity'):
    from sentence_transformers import SentenceTransformer, util
    import numpy as np
    import pandas as pd

    model = SentenceTransformer('all-MiniLM-L6-v2')

    df = df.copy()
    df['ref_group'] = (df[ref_col] != df[ref_col].shift()).cumsum()

    all_scores = {}

    for gid, group in df.groupby('ref_group'):
        ref_text = group[ref_col].iloc[0]

        # Skip entire group if reference text is missing or invalid
        if pd.isna(ref_text) or not isinstance(ref_text, str) or not ref_text.strip():
            for idx in group.index:
                all_scores[idx] = np.nan
            continue

        cmp_texts = group[cmp_col].tolist()
        cmp_indices = group.index.tolist()

        sim_scores = []
        for t in cmp_texts:
            if pd.isna(t) or not isinstance(t, str) or not t.strip():
                sim_scores.append(np.nan)
            else:
                emb_ref = model.encode(ref_text, convert_to_tensor=True)
                emb_cmp = model.encode(t, convert_to_tensor=True)
                score = util.cos_sim(emb_ref, emb_cmp).item()
                sim_scores.append(score)

        for i, idx in enumerate(cmp_indices):
            all_scores[idx] = sim_scores[i]

    df[score_col] = df.index.to_series().map(all_scores)
    return df.drop(columns=['ref_group'])

#BioBert & Clinical Bert

def load_bert_model(model_type='biobert'):
    if model_type == 'biobert':
        model_name = 'dmis-lab/biobert-base-cased-v1.1'
    elif model_type == 'clinicalbert':
        model_name = 'emilyalsentzer/Bio_ClinicalBERT'
    else:
        raise ValueError("Invalid model_type. Use 'biobert' or 'clinicalbert'.")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    return tokenizer, model

def get_sentence_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use [CLS] token representation
    return outputs.last_hidden_state[:, 0, :].squeeze()

def bert_row_similarity(row, ref_col, cmp_col, tokenizer, model):
    if pd.isna(row[ref_col]) or pd.isna(row[cmp_col]):
        return np.nan
    text1, text2 = row[ref_col], row[cmp_col]
    if not isinstance(text1, str) or not text1.strip() or not isinstance(text2, str) or not text2.strip():
        return np.nan
    emb1 = get_sentence_embedding(text1, tokenizer, model)
    emb2 = get_sentence_embedding(text2, tokenizer, model)
    return cosine_similarity(emb1, emb2, dim=0).item()

def bert_group_similarity(df, ref_col, cmp_col, tokenizer, model, score_col='BERT_group_similarity'):
    df = df.copy()
    df['ref_group'] = (df[ref_col] != df[ref_col].shift()).cumsum()

    all_scores = {}

    for gid, group in df.groupby('ref_group'):
        ref_text = group[ref_col].iloc[0]

        if pd.isna(ref_text) or not isinstance(ref_text, str) or not ref_text.strip():
            for idx in group.index:
                all_scores[idx] = np.nan
            continue

        ref_emb = get_sentence_embedding(ref_text, tokenizer, model)
        cmp_texts = group[cmp_col].tolist()
        cmp_indices = group.index.tolist()

        sim_scores = []
        for t in cmp_texts:
            if pd.isna(t) or not isinstance(t, str) or not t.strip():
                sim_scores.append(np.nan)
            else:
                cmp_emb = get_sentence_embedding(t, tokenizer, model)
                sim = torch_cosine_similarity(ref_emb, cmp_emb, dim=0).item()
                sim_scores.append(sim)

        for i, idx in enumerate(cmp_indices):
            all_scores[idx] = sim_scores[i]

    df[score_col] = df.index.to_series().map(all_scores)
    return df.drop(columns=['ref_group'])




In [40]:

# Path to your Excel file
file_path = "Similar Papers fetch V2.csv"  # Replace with your actual path if different

# Read the Excel file into a DataFrame
papers = pd.read_csv(file_path)

# Optional: display the first few rows
print(papers.head())


                                   Seed paper title   \
0  W4378417861: Relation of serum BDNF to major d...   
1  W4378417861: Relation of serum BDNF to major d...   
2  W4378417861: Relation of serum BDNF to major d...   
3  W4378417861: Relation of serum BDNF to major d...   
4  W4378417861: Relation of serum BDNF to major d...   

                          1st Jump Referenced Papers  \
0  W4378417861: Relation of serum BDNF to major d...   
1  W1759151863: Predictors, moderators, and media...   
2  W1830719945: Likelihood Ratio Tests for Model ...   
3  W1877874306: BDNF – a key transducer of antide...   
4  W1932805014: Brain-derived Neurotrophic Factor...   

                                   1st Jump Citation  
0  W4394751902: Effect of Regular Aerobic Exercis...  
1  W4401879508: Ginsenoside Re Prevents Depressio...  
2  W4410832792: Omega-3 fatty acids and fetal bra...  
3  W4410939997: The role of vitamin D in sleep re...  
4  W4392710855: Struggling Can Also Show on the I..

In [41]:
papers.head(5)

,Seed paper title,1st Jump Referenced Papers,1st Jump Citation
0,W4378417861: Relation of serum BDNF to major d...,W4378417861: Relation of serum BDNF to major d...,W4394751902: Effect of Regular Aerobic Exercis...
1,W4378417861: Relation of serum BDNF to major d...,"W1759151863: Predictors, moderators, and media...",W4401879508: Ginsenoside Re Prevents Depressio...
2,W4378417861: Relation of serum BDNF to major d...,W1830719945: Likelihood Ratio Tests for Model ...,W4410832792: Omega-3 fatty acids and fetal bra...
3,W4378417861: Relation of serum BDNF to major d...,W1877874306: BDNF – a key transducer of antide...,W4410939997: The role of vitamin D in sleep re...
4,W4378417861: Relation of serum BDNF to major d...,W1932805014: Brain-derived Neurotrophic Factor...,W4392710855: Struggling Can Also Show on the I...


In [42]:
papers.iloc[0]["1st Jump Referenced Papers"]

'W4378417861: Relation of serum BDNF to major depression and exploration of mechanistic roles of serum BDNF in a study of vitamin D3 and omega-3 supplements for late-life depression prevention'

In [43]:
papers.columns

Index(['Seed paper title ', '1st Jump Referenced Papers', '1st Jump Citation'], dtype='object')

In [44]:
papers["Simple_Similarity_referenced_papers"] = papers.apply(
    lambda row: simple_similarity(row['Seed paper title '], row['1st Jump Referenced Papers'])
    if pd.notna(row['Seed paper title ']) and pd.notna(row['1st Jump Referenced Papers'])
    else np.nan,
    axis=1
)


papers["Simple_Similarity_citation_papers"] = papers.apply(
    lambda row: simple_similarity(row['Seed paper title '], row['1st Jump Citation'])
    if pd.notna(row['Seed paper title ']) and pd.notna(row['1st Jump Citation'])
    else np.nan,
    axis=1
)

In [45]:
# Create new column using corpus-wide TF-IDF grouped by changing reference
papers = compute_tfidf_scores_apply(
    df=papers,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Referenced Papers',
    score_col='TFIDF score on referenced papers'
)

In [46]:
papers = compute_tfidf_scores_apply(
    df=papers,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Citation',
    score_col='TFIDF score on citation'
)


In [47]:
papers.columns

Index(['Seed paper title ', '1st Jump Referenced Papers', '1st Jump Citation',
       'Simple_Similarity_referenced_papers',
       'Simple_Similarity_citation_papers', 'TFIDF score on referenced papers',
       'TFIDF score on citation'],
      dtype='object')

In [48]:
papers_v2 = papers[['Seed paper title ','1st Jump Referenced Papers','Simple_Similarity_referenced_papers','TFIDF score on referenced papers',
                    '1st Jump Citation','Simple_Similarity_citation_papers','TFIDF score on citation']]

In [49]:
#papers_v2.to_csv("score_tf_idf_V5.csv")

In [50]:
#papers_v2.loc[:, "SBERT_Referenced_row_similarity"] = papers_v2.apply(
#    lambda row: sbert_row_similarity(row, 'Seed paper title ', '1st Jump Referenced Papers'),
#    axis=1
#)

In [51]:
papers_v2

,Seed paper title,1st Jump Referenced Papers,Simple_Similarity_referenced_papers,TFIDF score on referenced papers,1st Jump Citation,Simple_Similarity_citation_papers,TFIDF score on citation
0,W4378417861: Relation of serum BDNF to major d...,W4378417861: Relation of serum BDNF to major d...,1.000000,1.000000,W4394751902: Effect of Regular Aerobic Exercis...,0.100000,0.134978
1,W4378417861: Relation of serum BDNF to major d...,"W1759151863: Predictors, moderators, and media...",0.090909,0.085799,W4401879508: Ginsenoside Re Prevents Depressio...,0.071429,0.084807
2,W4378417861: Relation of serum BDNF to major d...,W1830719945: Likelihood Ratio Tests for Model ...,0.000000,0.000000,W4410832792: Omega-3 fatty acids and fetal bra...,0.074074,0.118185
3,W4378417861: Relation of serum BDNF to major d...,W1877874306: BDNF – a key transducer of antide...,0.052632,0.081846,W4410939997: The role of vitamin D in sleep re...,0.090909,0.099760
4,W4378417861: Relation of serum BDNF to major d...,W1932805014: Brain-derived Neurotrophic Factor...,0.000000,0.000000,W4392710855: Struggling Can Also Show on the I...,0.000000,0.000000
...,...,...,...,...,...,...,...
334,W4406696697: The risks of adverse events with ...,W4361280733: The risks of adverse events with ...,0.636364,0.584474,NaN,NaN,0.000000
335,W4406696697: The risks of adverse events with ...,W4388532888: Pharmacological update of mirtaza...,0.111111,0.083382,NaN,NaN,0.000000
336,W4406696697: The risks of adverse events with ...,W4391112466: Beneficial and harmful effects of...,0.526316,0.429806,NaN,NaN,0.000000
337,W4406696697: The risks of adverse events with ...,W4396699590: Canadian Network for Mood and Anx...,0.081633,0.083746,NaN,NaN,0.000000


In [52]:
papers_v2 = sbert_group_similarity(
    df=papers_v2,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Referenced Papers',
    score_col='SBERT_Referenced_Paper_similarity'
)


papers_v2 = sbert_group_similarity(
    df=papers_v2,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Citation',
    score_col='SBERT_Citation_Paper_similarity'
)

/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [53]:
# Choose model type
tokenizer, model = load_bert_model(model_type='biobert')  # or 'clinicalbert'

# OR apply group-wise
papers_v2 = bert_group_similarity(
    df=papers_v2,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Referenced Papers',
    tokenizer=tokenizer,
    model=model,
    score_col='BioBERT_Referenced_group_similarity'
)

papers_v2 = bert_group_similarity(
    df=papers_v2,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Citation',
    tokenizer=tokenizer,
    model=model,
    score_col='BioBERT_Citation_similarity'
)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [54]:
# Choose model type
tokenizer, model = load_bert_model(model_type='clinicalbert')  # or 'clinicalbert'

# OR apply group-wise
papers_v2 = bert_group_similarity(
    df=papers_v2,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Referenced Papers',
    tokenizer=tokenizer,
    model=model,
    score_col='ClinicalBERT_Referenced_group_similarity'
)

papers_v2 = bert_group_similarity(
    df=papers_v2,
    ref_col='Seed paper title ',
    cmp_col='1st Jump Citation',
    tokenizer=tokenizer,
    model=model,
    score_col='ClinicalBERT_Citation_similarity'
)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [55]:
papers_v3 = papers_v2[['Seed paper title ','1st Jump Referenced Papers','Simple_Similarity_referenced_papers',
                       'TFIDF score on referenced papers','SBERT_Referenced_Paper_similarity',
                       'BioBERT_Referenced_group_similarity', 'ClinicalBERT_Referenced_group_similarity',
                    '1st Jump Citation','Simple_Similarity_citation_papers','TFIDF score on citation',
                       'SBERT_Citation_Paper_similarity','BioBERT_Citation_similarity','ClinicalBERT_Citation_similarity'
                      ]]

In [56]:
papers_v3

,Seed paper title,1st Jump Referenced Papers,Simple_Similarity_referenced_papers,TFIDF score on referenced papers,SBERT_Referenced_Paper_similarity,BioBERT_Referenced_group_similarity,ClinicalBERT_Referenced_group_similarity,1st Jump Citation,Simple_Similarity_citation_papers,TFIDF score on citation,SBERT_Citation_Paper_similarity,BioBERT_Citation_similarity,ClinicalBERT_Citation_similarity
0,W4378417861: Relation of serum BDNF to major d...,W4378417861: Relation of serum BDNF to major d...,1.000000,1.000000,1.000000,1.000000,1.000000,W4394751902: Effect of Regular Aerobic Exercis...,0.100000,0.134978,0.467744,0.960881,0.946107
1,W4378417861: Relation of serum BDNF to major d...,"W1759151863: Predictors, moderators, and media...",0.090909,0.085799,0.452280,0.904760,0.916082,W4401879508: Ginsenoside Re Prevents Depressio...,0.071429,0.084807,0.491517,0.908716,0.939275
2,W4378417861: Relation of serum BDNF to major d...,W1830719945: Likelihood Ratio Tests for Model ...,0.000000,0.000000,0.083601,0.905982,0.920828,W4410832792: Omega-3 fatty acids and fetal bra...,0.074074,0.118185,0.527439,0.930938,0.934114
3,W4378417861: Relation of serum BDNF to major d...,W1877874306: BDNF – a key transducer of antide...,0.052632,0.081846,0.615017,0.918118,0.922206,W4410939997: The role of vitamin D in sleep re...,0.090909,0.099760,0.455187,0.929830,0.917251
4,W4378417861: Relation of serum BDNF to major d...,W1932805014: Brain-derived Neurotrophic Factor...,0.000000,0.000000,0.317847,0.919805,0.900901,W4392710855: Struggling Can Also Show on the I...,0.000000,0.000000,0.343499,0.942600,0.947340
...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,W4406696697: The risks of adverse events with ...,W4361280733: The risks of adverse events with ...,0.636364,0.584474,0.849894,0.972619,0.983957,NaN,NaN,0.000000,NaN,NaN,NaN
335,W4406696697: The risks of adverse events with ...,W4388532888: Pharmacological update of mirtaza...,0.111111,0.083382,0.678012,0.938321,0.952534,NaN,NaN,0.000000,NaN,NaN,NaN
336,W4406696697: The risks of adverse events with ...,W4391112466: Beneficial and harmful effects of...,0.526316,0.429806,0.631931,0.985816,0.983894,NaN,NaN,0.000000,NaN,NaN,NaN
337,W4406696697: The risks of adverse events with ...,W4396699590: Canadian Network for Mood and Anx...,0.081633,0.083746,0.514838,0.838343,0.891523,NaN,NaN,0.000000,NaN,NaN,NaN


In [57]:
#papers_v3.to_csv("score_tfidf_symantic V2.csv")

In [37]:
MDD_Word_Cloud = ["5-HT1A", "5-HT4R", "Agomelatine", "AMPA Receptor Activation", "AMPA antagonist", "Analgesics", "Anesthetics",
    "Antidepressant", "Antidepressants", "Antipsychotic", "Antipsychotics", "Ayahuasca", "BDI", "BDNF", "Beta-arrestin",
    "Biomarker", "Biomarkers", "Brexpiprazole", "Bupropion", "Citalopram", "Clomipramine", "CYP2C19", "CYP2D6",
    "Central Nervous System Depressants", "Cyclooxygenase Inhibitors", "Desvenlafaxine", "Dopamine Reuptake Inhibitor",
    "Depression", "Depress", "Depressive", "MDD", "Duloxetine", "EEG", "Escitalopram", "Estrogen Antagonists", 
    "Excitatory Amino Acid Agents", "Excitatory Amino Acid Antagonists",
    "Fluoxetine", "GABA Agents", "Gene", "GeneSight", "Genetic", "Genetics", "Genomics", "General Anesthetics",
    "Glutamate", "Glutamate antagonist", "HAMD", "HAM-D", "HDRS", "Hydroxynorketamine", "IGF-1", "Inventory (generic)",
    "Ketamine", "Leucine", "MADRS", "MDMA", "MTHFR", "Magnetic Resonance Spectroscopy", "Magnetoencephalography",
    "Metabotropic glutamate receptor antagonist", "Mirtazapine", "Montgomery-Åsberg Depression Rating Scale",
    "NEGR1", "NMDA antagonist", "NMDA receptor antagonist", "Norepinephrine Reuptake Inhibitor", "Opioid",
    "PACAP", "PHQ", "PHQ-8", "PHQ-9", "PRKCB1", "Paroxetine", "Patient Health Questionnaire", "Peripheral Nervous System Agents",
    "Psilocybin", "P2X7", "QEEG", "QIDS", "SERT", "SNRI", "SNRIs", "SSRI", "SSRIs", "SV2A", "Selective Serotonin Reuptake Inhibitor",
    "Sensory System Agents", "Serotonin Uptake Inhibitors", "Serotonin transporter gene polymorphism",
    "SPECT", "Trazodone", "Thyroid pathway genes", "Triiodothyronine", "Venlafaxine", "Vilazodone", "Vortioxetine"
]


In [62]:
papers_v3["Word_Cloud_Reference_Similarity"] = papers_v3["1st Jump Referenced Papers"].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

papers_v3["Word_Cloud_Citation_Similarity"] = papers_v3["1st Jump Citation"].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_48648/3998050552.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  papers_v3["Word_Cloud_Reference_Similarity"] = papers_v3["1st Jump Referenced Papers"].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_48648/3998050552.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  papers_v3["Word_Cloud_Citation_Similarity"] = papers_v3["1st Jump Citation"].apply(lambda

In [64]:
papers_v3 = papers_v3[['Seed paper title ','1st Jump Referenced Papers','Simple_Similarity_referenced_papers',
                       'TFIDF score on referenced papers','SBERT_Referenced_Paper_similarity',
                       'BioBERT_Referenced_group_similarity', 'ClinicalBERT_Referenced_group_similarity',
                       'Word_Cloud_Reference_Similarity', 
                        '1st Jump Citation','Simple_Similarity_citation_papers','TFIDF score on citation',
                       'SBERT_Citation_Paper_similarity','BioBERT_Citation_similarity','ClinicalBERT_Citation_similarity',
                       'Word_Cloud_Citation_Similarity'
                      ]]

In [67]:
papers_v3.to_csv("score_tfidf_symantic V3.csv")